# Physics-guided features: a cantilever surrogate

**Learning objectives:** fit a regression model to finite-element data; compare held-out predictions; explain how a physical scaling law changes feature choice.

Prerequisites: basic linear regression and linear elasticity. Requirements: NumPy and Matplotlib; run the cells in order. No FEM installation or external download is needed.

Author: Haoming Luo (AgentFEM maintainer). Code: Apache-2.0.


## 1. Mechanical problem and data

![Cantilever geometry and loading schematic](assets/cantilever-setup.png)

A 1 m × 0.2 m plane-strain cantilever is fixed along its left edge and carries a uniform downward traction of 1 MPa on its right edge. Poisson ratio is 0.30. A 20 × 4 quadrilateral mesh with first-order displacement was used. Only Young's modulus varies, from 150 to 250 GPa.

The response $d$ is the **global maximum absolute displacement degree of freedom**, in mm, not a displacement-magnitude field. The ten synthetic FEM results below are not laboratory measurements. The schematic shows geometry and loading, not computed deformation.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

data = np.array([
    [236.527848027, 0.379486114682],
    [183.705005271, 0.488604183265],
    [221.773531918, 0.404732851957],
    [207.905182459, 0.431730623549],
    [219.051438367, 0.409762358700],
    [151.789348137, 0.591339479113],
    [193.549173343, 0.463753125427],
    [242.983027674, 0.369404542042],
    [174.672684011, 0.513869896544],
    [166.399131657, 0.539420086920],
])
young_GPa, displacement_mm = data.T
indices = np.random.default_rng(2026).permutation(len(data))
test, train = indices[:2], indices[2:]
print(f"{len(train)} training cases; {len(test)} held-out cases")

fig, ax = plt.subplots(figsize=(9, 3))
ax.add_patch(Rectangle((0, 0), 1, 0.2, facecolor="#e3edf0", edgecolor="#29495b"))
ax.add_patch(Rectangle((-0.035, -0.025), 0.035, 0.25, hatch="////", facecolor="white", edgecolor="#29495b"))
for y in np.linspace(0.04, 0.19, 3):
    ax.annotate("", xy=(1.01, y - 0.045), xytext=(1.01, y), arrowprops=dict(arrowstyle="->", color="#ad5b25"))
ax.annotate("", xy=(1, -0.08), xytext=(0, -0.08), arrowprops=dict(arrowstyle="<->"))
ax.text(0.5, -0.1, "1 m", ha="center", va="top")
ax.text(-0.06, 0.1, "Fixed", ha="right", va="center")
ax.text(0.5, 0.23, "Plane strain; height 0.2 m; Poisson ratio 0.30", ha="center")
ax.text(1.04, 0.1, "Uniform traction\n1 MPa downward", va="center")
ax.set(xlim=(-0.16, 1.5), ylim=(-0.16, 0.32))
ax.set_aspect("equal")
ax.axis("off")
fig.tight_layout()
plt.show()


## 2. Same learner, different features

At fixed geometry, mesh and Poisson ratio, the linear-elastic stiffness satisfies $K(E)=E K_0$. With fixed loads, $E K_0 u=f$, hence $u\propto 1/E$ and $d\propto 1/E$.

We fit two coefficients in each model using the **same eight training cases**:

- Model A: $\hat d=a+b(E-150)/100$.
- Model B: $\hat d=a+b(210/E)$.

Here E is expressed in GPa; scaling improves numerical conditioning and uses a declared range, not test-set statistics. Neither fit uses the two held-out responses. Before running, predict which model should work better and why.


In [ ]:
def features(e, mode):
    e = np.atleast_1d(np.asarray(e, dtype=float))
    if mode == "linear_E":
        x = (e - 150.0) / 100.0
    elif mode == "inverse_E":
        x = 210.0 / e
    else:
        raise ValueError("Unknown feature")
    return np.column_stack((np.ones_like(e), x))

models = {}
for mode in ("linear_E", "inverse_E"):
    weights = np.linalg.lstsq(features(young_GPa[train], mode), displacement_mm[train], rcond=None)[0]
    models[mode] = weights
    prediction = features(young_GPa[test], mode) @ weights
    error = np.linalg.norm(prediction - displacement_mm[test]) / np.linalg.norm(displacement_mm[test])
    print(f"{mode:10s}: held-out relative L2 = {100 * error:.6g}%")
    for e, actual, pred in zip(young_GPa[test], displacement_mm[test], prediction):
        print(f"  E={e:.3f} GPa; FEM={actual:.6f} mm; prediction={pred:.6f} mm")


In [ ]:
grid = np.linspace(150, 250, 200)
colors = {"linear_E": "#16857f", "inverse_E": "#7356a6"}
labels = {"linear_E": "Model A: E", "inverse_E": "Model B: 1/E"}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ax = axes[0]
ax.scatter(young_GPa[train], displacement_mm[train], color="#29495b", label="8 training FEM cases", zorder=3)
ax.scatter(young_GPa[test], displacement_mm[test], color="#ad5b25", marker="D", s=65, label="2 held-out FEM cases", zorder=4)
for mode, weights in models.items():
    ax.plot(grid, features(grid, mode) @ weights, color=colors[mode],
            linestyle="--" if mode == "inverse_E" else "-", label=labels[mode])
ax.set(xlabel="Young's modulus / GPa", ylabel="Maximum absolute displacement DOF / mm",
       title="Data and fitted models")
ax.legend(fontsize=8)
ax = axes[1]
for mode, weights in models.items():
    residual_um = 1000 * (features(young_GPa[test], mode) @ weights - displacement_mm[test])
    order = np.argsort(young_GPa[test])
    ax.plot(young_GPa[test][order], residual_um[order], "o", color=colors[mode], label=labels[mode])
ax.axhline(0, color="#777777", linewidth=0.8)
ax.set(xlabel="Held-out Young's modulus / GPa", ylabel="Prediction minus FEM / micrometre",
       title="Held-out errors (two cases)")
ax.legend(fontsize=8)
for ax in axes:
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()


## 3. Explore and interpret

![Reference plot: fitted models and held-out errors](assets/cantilever-models.png)

Reference preview for the default split; running the cell above regenerates it.


1. Query E = 180, 210 and 230 GPa. These are model estimates, not new FEM computations.
2. Set a different split seed in the first cell and rerun. Does the physical explanation change?
3. Why is a nearly exact fit possible with just eight samples here? Would the argument survive changing geometry, contact conditions or plastic deformation?

The inverse-E fit is expected from the governing equations, not evidence of a generally superior learner. Two held-out cases cannot establish broad predictive reliability. This exercise teaches representation and verification rather than a general-purpose engineering surrogate.


In [ ]:
query_GPa = 210.0
if not 150 <= query_GPa <= 250:
    raise ValueError("Outside the teaching range; additional FEM data are needed.")
for mode, weights in models.items():
    response = float((features([query_GPa], mode) @ weights)[0])
    print(f"{mode:10s}: E={query_GPa:g} GPa -> predicted d={response:.6f} mm")


## Data provenance and further reading

These values are unchanged from the [published dataset and provenance record](https://gist.github.com/haoming-luo/727a217b5b2161a4e81f4ec65aacfc56). Generated on 2026-09-20 with AgentFEM 0.3.3 and DOLFINx 0.11.0; Latin-hypercube sampling, seed 2026.

- Source script: `examples/static_elasticity_surrogate_campaign.py` at commit `08f3ca5a871aff5084142837f3030e0a2d88dfd5` in [AgentFEM](https://github.com/haoming-luo/agentfem).
- Executed script SHA-256: `feda6575ab4b3c9c13a3995ea2b450ae3127632561ff2d6e489e8db0c4447853`.
- Original CSV SHA-256: `8d4320fcbee11955dc96f720dfaaded33c964bd63f1398bc2dd6cad1ac5b5600` (hash refers to the original file bytes, not the embedded array).
- The source commit identifies the example, not the installed package's source revision.

[Complete FEM-to-surrogate workflow](https://haoming-luo.github.io/agentfem/examples/simulation_to_surrogate/). This notebook independently fits NumPy regressions; it does not execute AgentFEM's built-in estimator or a neural operator.
